<a href="https://colab.research.google.com/github/ekomissarov/demos/blob/main/one_sql_ztest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Python libs
import pandas as pd

# Magics
from helpers import (
    load_sql_magic,
    load_viz_magic
)
load_sql_magic()          # %%sql   — query DataFrames via duckdb (no extra installs)
load_viz_magic()          # %%viz   — open a DataFrame in PyGWalker (drag-and-drop charts)

True

In [ ]:
from __future__ import annotations

import json
import math
import random

from dataclasses import dataclass, field
from datetime import date, datetime, time, timedelta
from typing import Iterable, Optional

import pandas as pd
import xxhash


@dataclass(frozen=True)
class GenConfig:
    start_date: date
    end_date: date
    users: int
    seed: Optional[int] = 42

    # Event names
    event_page_view: str = "page_view"
    event_watch: str = "watch"
    event_add_to_cart: str = "add_to_cart"
    event_purchase: str = "purchase"

    # Countries
    countries: tuple[str, ...] = ("US", "GB", "DE")
    country_weights: tuple[float, ...] = (0.55, 0.25, 0.20)

    # Seasonality / traffic shape (Mon..Sun)
    dow_multipliers: tuple[float, ...] = (0.95, 1.00, 1.02, 1.03, 1.05, 1.20, 1.10)
    day_noise_sigma: float = 0.08

    # Sessions
    base_sessions_per_user_per_day: float = 0.35

    # PV per session
    pv_logn_mu: float = 1.2
    pv_logn_sigma: float = 0.55
    pv_cap: int = 35

    # Funnel baseline
    p_watch_given_pv: float = 0.25
    p_atc_given_watch: float = 0.18
    p_purchase_given_atc: float = 0.22
    p_purchase_given_watch_no_atc: float = 0.04
    p_purchase_without_watch: float = 0.004

    # Experiment effect
    exp_b_mult_purchase_given_atc: float = 1.18
    exp_b_mult_watch_given_pv: float = 1.12

    # Purchase amount
    amount_mu: float = 3.6
    amount_sigma: float = 0.55
    p_high_ticket: float = 0.06
    high_ticket_mu: float = 5.3
    high_ticket_sigma: float = 0.35
    amount_min: float = 1.0
    amount_max: float = 5000.0

    # Staggered adoption shock
    staggered_start_dates: dict[str, Optional[date]] = field(
        default_factory=lambda: {
            "GB": date(2025, 2, 1),
            "DE": date(2025, 2, 15),
            "US": None,
        }
    )

    staggered_purchase_mult: float = 1.35


class RNG:
    def __init__(self, seed: Optional[int]) -> None:
        self._r = random.Random(seed)

    def random(self) -> float:
        return self._r.random()

    def randint(self, a: int, b: int) -> int:
        return self._r.randint(a, b)

    def randrange(self, a: int, b: Optional[int] = None) -> int:
        return self._r.randrange(a) if b is None else self._r.randrange(a, b)

    def gauss(self, mu: float, sigma: float) -> float:
        return self._r.gauss(mu, sigma)

    def lognorm(self, mu: float, sigma: float) -> float:
        return self._r.lognormvariate(mu, sigma)

    def bern(self, p: float) -> bool:
        return self._r.random() < p

    def choice_weighted(self, items: tuple[str, ...], weights: tuple[float, ...]) -> str:
        total = sum(weights)
        x = self._r.random() * total
        acc = 0.0

        for item, w in zip(items, weights):
            acc += w
            if x <= acc:
                return item

        return items[-1]

    def poisson(self, lmbd: float) -> int:
        if lmbd <= 0:
            return 0

        L = math.exp(-lmbd)
        k = 0
        p = 1.0

        while p > L:
            k += 1
            p *= self._r.random()

        return k - 1


class AbEventsGenerator:
    def __init__(self, cfg: GenConfig) -> None:
        self.cfg = cfg
        self.rng = RNG(cfg.seed)

    @staticmethod
    def daterange(d1: date, d2: date) -> Iterable[date]:
        cur = d1
        while cur <= d2:
            yield cur
            cur += timedelta(days=1)

    @staticmethod
    def user_group_ab(user_id: int) -> str:
        h = xxhash.xxh64(f"user_with_id_{user_id}", seed=0).intdigest()
        return "a" if h % 2 == 0 else "b"

    @classmethod
    def user_experiment_json(cls, user_id: int) -> str:
        return json.dumps(
            {"num01": cls.user_group_ab(user_id)},
            ensure_ascii=False,
            separators=(",", ":"),
        )

    @staticmethod
    def _clamp(x: float, lo: float, hi: float) -> float:
        return lo if x < lo else hi if x > hi else x

    def _day_multiplier(self, d: date) -> float:
        cfg = self.cfg
        base = cfg.dow_multipliers[d.weekday()]

        h = xxhash.xxh64(f"daynoise:{d.isoformat()}", seed=123).intdigest() % 1_000_000
        u = (h / 1_000_000.0) * 2 - 1
        noise = math.exp(u * cfg.day_noise_sigma)

        return base * noise

    @staticmethod
    def _segment(user_id: int) -> str:
        h = xxhash.xxh64(f"seg:{user_id}", seed=17).intdigest() % 10_000
        x = h / 10_000.0

        if x < 0.72:
            return "casual"
        if x < 0.95:
            return "regular"
        return "power"

    def _random_time_in_day(self, d: date, segment: str) -> datetime:
        r = self.rng.random()

        if segment == "casual":
            mu, sigma = (20.0, 2.2) if r < 0.75 else (13.0, 3.5)

        elif segment == "power":
            if r < 0.20:
                mu, sigma = 9.0, 1.8
            elif r < 0.70:
                mu, sigma = 14.0, 3.0
            else:
                mu, sigma = 20.0, 2.5

        else:
            mu, sigma = (14.0, 3.5) if r < 0.7 else (20.0, 2.0)

        hour = int(min(23, max(0, self.rng.gauss(mu, sigma))))
        minute = self.rng.randrange(0, 60)
        second = self.rng.randrange(0, 60)

        return datetime.combine(d, time(hour=hour, minute=minute, second=second))

    def _inter_event_delay_sec(self, kind: str) -> int:
        if kind == "pv":
            return self.rng.randint(4, 70)
        if kind == "watch":
            return self.rng.randint(10, 220)
        if kind == "atc":
            return self.rng.randint(8, 160)
        if kind == "purchase":
            return self.rng.randint(12, 300)

        return self.rng.randint(5, 120)

    def _pageviews_per_session(self, segment: str) -> int:
        cfg = self.cfg
        base = self.rng.lognorm(cfg.pv_logn_mu, cfg.pv_logn_sigma)
        mult = 0.85 if segment == "casual" else 1.0 if segment == "regular" else 1.35

        n = int(round(base * mult))
        return min(cfg.pv_cap, max(1, n))

    def _amount(self, segment: str, country: str) -> float:
        cfg = self.cfg

        p_hi = cfg.p_high_ticket * (
            1.25 if segment == "power" else 0.9 if segment == "casual" else 1.0
        )

        if self.rng.bern(p_hi):
            amt = self.rng.lognorm(cfg.high_ticket_mu, cfg.high_ticket_sigma)
        else:
            amt = self.rng.lognorm(cfg.amount_mu, cfg.amount_sigma)

        amt *= 1.08 if country == "US" else 1.00 if country == "GB" else 0.95
        amt = self._clamp(amt, cfg.amount_min, cfg.amount_max)

        return round(amt, 2)

    def _apply_experiment_effects(
        self,
        uid: int,
        p_watch: float,
        p_purch_atc: float,
    ) -> tuple[float, float]:
        cfg = self.cfg

        if self.user_group_ab(uid) == "b":
            p_watch *= cfg.exp_b_mult_watch_given_pv
            p_purch_atc *= cfg.exp_b_mult_purchase_given_atc

        return p_watch, p_purch_atc

    @staticmethod
    def _to_int63(u64: int) -> int:
        return u64 & ((1 << 63) - 1)

    def generate(self) -> pd.DataFrame:
        cfg = self.cfg

        columns = ["date", "user_id", "hash_id", "country", "experiment", "event_type", "amount"]

        user_country = {}
        user_segment = {}
        user_experiment = {}
        user_hash_id = {}

        for uid in range(1, cfg.users + 1):
            user_country[uid] = self.rng.choice_weighted(cfg.countries, cfg.country_weights)
            user_segment[uid] = self._segment(uid)
            user_experiment[uid] = self.user_experiment_json(uid)

            raw_u64 = xxhash.xxh64(str(uid).encode()).intdigest()
            user_hash_id[uid] = self._to_int63(raw_u64)

        rows = []

        for d in self.daterange(cfg.start_date, cfg.end_date):
            day_mult = self._day_multiplier(d)

            for uid in range(1, cfg.users + 1):
                country = user_country[uid]
                segment = user_segment[uid]
                experiment = user_experiment[uid]
                hash_id = user_hash_id[uid]

                base_activity = 0.7 if segment == "casual" else 1.0 if segment == "regular" else 1.8
                funnel_mult = 0.85 if segment == "casual" else 1.0 if segment == "regular" else 1.25
                repurchase_mult = 0.8 if segment == "casual" else 1.0 if segment == "regular" else 1.4

                if country == "DE":
                    funnel_mult *= 0.95
                elif country == "US":
                    funnel_mult *= 1.03

                lmbd = cfg.base_sessions_per_user_per_day * base_activity * day_mult
                sessions = self.rng.poisson(lmbd)

                if sessions <= 0:
                    continue

                for _ in range(sessions):
                    tcur = self._random_time_in_day(d, segment)
                    n_pv = self._pageviews_per_session(segment)

                    p_watch = cfg.p_watch_given_pv * funnel_mult
                    p_atc = cfg.p_atc_given_watch * funnel_mult
                    p_purch_atc = cfg.p_purchase_given_atc * funnel_mult
                    p_purch_watch = cfg.p_purchase_given_watch_no_atc * funnel_mult
                    p_purch_wo_watch = cfg.p_purchase_without_watch * repurchase_mult

                    treatment_start = cfg.staggered_start_dates.get(country)

                    if treatment_start is not None and d >= treatment_start:
                        p_purch_atc *= cfg.staggered_purchase_mult
                        p_purch_watch *= cfg.staggered_purchase_mult
                        p_purch_wo_watch *= cfg.staggered_purchase_mult

                    p_watch, p_purch_atc = self._apply_experiment_effects(uid, p_watch, p_purch_atc)

                    did_watch = False
                    did_atc = False
                    did_purchase = False
                    allow_second_purchase = segment == "power" and self.rng.bern(0.06)

                    for _ in range(n_pv):
                        tcur += timedelta(seconds=self._inter_event_delay_sec("pv"))
                        rows.append(
                            (tcur.isoformat(sep=" "), uid, hash_id, country, experiment, cfg.event_page_view, None)
                        )

                        if not did_purchase and self.rng.bern(p_purch_wo_watch):
                            tcur += timedelta(seconds=self._inter_event_delay_sec("purchase"))
                            rows.append(
                                (
                                    tcur.isoformat(sep=" "),
                                    uid,
                                    hash_id,
                                    country,
                                    experiment,
                                    cfg.event_purchase,
                                    self._amount(segment, country),
                                )
                            )
                            did_purchase = True

                        if not did_watch and self.rng.bern(p_watch):
                            tcur += timedelta(seconds=self._inter_event_delay_sec("watch"))
                            rows.append(
                                (tcur.isoformat(sep=" "), uid, hash_id, country, experiment, cfg.event_watch, None)
                            )
                            did_watch = True

                            if not did_atc and self.rng.bern(p_atc):
                                tcur += timedelta(seconds=self._inter_event_delay_sec("atc"))
                                rows.append(
                                    (
                                        tcur.isoformat(sep=" "),
                                        uid,
                                        hash_id,
                                        country,
                                        experiment,
                                        cfg.event_add_to_cart,
                                        None,
                                    )
                                )
                                did_atc = True

                                if self.rng.bern(p_purch_atc):
                                    tcur += timedelta(seconds=self._inter_event_delay_sec("purchase"))
                                    rows.append(
                                        (
                                            tcur.isoformat(sep=" "),
                                            uid,
                                            hash_id,
                                            country,
                                            experiment,
                                            cfg.event_purchase,
                                            self._amount(segment, country),
                                        )
                                    )
                                    did_purchase = True

                                    if allow_second_purchase and self.rng.bern(0.35):
                                        delay = self._inter_event_delay_sec("purchase") + self.rng.randint(30, 240)
                                        tcur += timedelta(seconds=delay)
                                        rows.append(
                                            (
                                                tcur.isoformat(sep=" "),
                                                uid,
                                                hash_id,
                                                country,
                                                experiment,
                                                cfg.event_purchase,
                                                self._amount(segment, country),
                                            )
                                        )

                            if not did_purchase and self.rng.bern(p_purch_watch):
                                tcur += timedelta(seconds=self._inter_event_delay_sec("purchase"))
                                rows.append(
                                    (
                                        tcur.isoformat(sep=" "),
                                        uid,
                                        hash_id,
                                        country,
                                        experiment,
                                        cfg.event_purchase,
                                        self._amount(segment, country),
                                    )
                                )
                                did_purchase = True

        return pd.DataFrame(rows, columns=columns)

In [ ]:
cfg = GenConfig(
    start_date=date(2025, 1, 1),
    end_date=date(2025, 2, 28),
    users=25_000,
    seed=42,
    staggered_start_dates={
        "GB": date(2025, 2, 1),
        "DE": date(2025, 2, 15),
        "US": None,
    },
    staggered_purchase_mult=1.35,
)

gen = AbEventsGenerator(cfg)
df = gen.generate()

df #.head()

,date,user_id,hash_id,country,experiment,event_type,amount
0,2025-01-01 10:32:46,2,6927017134761466251,US,"{""num01"":""a""}",page_view,NaN
1,2025-01-01 10:33:32,2,6927017134761466251,US,"{""num01"":""a""}",page_view,NaN
2,2025-01-01 21:27:17,4,1241045378047175561,US,"{""num01"":""b""}",page_view,NaN
3,2025-01-01 21:30:00,4,1241045378047175561,US,"{""num01"":""b""}",watch,NaN
4,2025-01-01 21:30:55,4,1241045378047175561,US,"{""num01"":""b""}",page_view,NaN
...,...,...,...,...,...,...,...
1995146,2025-02-28 20:40:52,24996,968404688415237291,GB,"{""num01"":""a""}",watch,NaN
1995147,2025-02-28 16:19:09,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN
1995148,2025-02-28 16:20:17,25000,4860591953457782041,GB,"{""num01"":""a""}",page_view,NaN
1995149,2025-02-28 16:20:30,25000,4860591953457782041,GB,"{""num01"":""a""}",watch,NaN


In [ ]:
%%sql result <<


-- z-test асимптотический нормальное приближение для разности средних (large-sample z-test),
-- а relative effect и его CI считаются отдельно через Delta Method.

WITH params AS (
    SELECT
        TIMESTAMP '2025-01-01 00:00:00' AS start_date,
        TIMESTAMP '2025-02-28 23:59:59' AS end_date,

        'num01' AS experiment_id,
        'a' AS control_group,
        'b' AS treatment_group,

        'purchase' AS metric
),

user_level AS (
    SELECT
        e.user_id,

        CASE
            WHEN e.experiment = '{"' || p.experiment_id || '":"' || p.control_group || '"}'
                THEN p.control_group
            WHEN e.experiment = '{"' || p.experiment_id || '":"' || p.treatment_group || '"}'
                THEN p.treatment_group
        END AS exp_group,

        -- User-level metric:
        -- X_i = total number of target events for user i over the whole period
        SUM(
            CASE
                WHEN e.event_type = p.metric THEN 1
                ELSE 0
            END
        ) AS metric_value

    FROM df e
    CROSS JOIN params p

    WHERE
        CAST(e.date AS TIMESTAMP) BETWEEN p.start_date AND p.end_date

        AND e.experiment IN (
            '{"' || p.experiment_id || '":"' || p.control_group || '"}',
            '{"' || p.experiment_id || '":"' || p.treatment_group || '"}'
        )

    GROUP BY
        e.user_id,
        exp_group
),

group_stats AS (
    SELECT
        exp_group,
        COUNT(*) AS users,
        SUM(metric_value) AS events,

        -- Sample mean:
        -- mean_metric = X̄
        AVG(metric_value * 1.0) AS mean_metric,

        -- Sample variance:
        -- var_metric = s²
        VAR_SAMP(metric_value * 1.0) AS var_metric

    FROM user_level
    GROUP BY exp_group
),

stats AS (
    SELECT
        MAX(CASE WHEN g.exp_group = p.control_group THEN g.events END) AS events_control,
        MAX(CASE WHEN g.exp_group = p.treatment_group THEN g.events END) AS events_treatment,

        MAX(CASE WHEN g.exp_group = p.control_group THEN g.users END) AS users_control,
        MAX(CASE WHEN g.exp_group = p.treatment_group THEN g.users END) AS users_treatment,

        MAX(CASE WHEN g.exp_group = p.control_group THEN g.mean_metric END) AS mean_control,
        MAX(CASE WHEN g.exp_group = p.treatment_group THEN g.mean_metric END) AS mean_treatment,

        MAX(CASE WHEN g.exp_group = p.control_group THEN g.var_metric END) AS var_control,
        MAX(CASE WHEN g.exp_group = p.treatment_group THEN g.var_metric END) AS var_treatment

    FROM group_stats g
    CROSS JOIN params p
),

calc AS (
    SELECT
        *,

        -- Relative effect:
        -- uplift = X̄_treatment / X̄_control - 1
        mean_treatment / NULLIF(mean_control, 0) - 1 AS relative_effect,

        -- Standard error of the relative effect via Delta Method:
        --
        -- g(X̄_treatment, X̄_control) = X̄_treatment / X̄_control - 1
        --
        -- Var(g) ≈
        --   s²_treatment / (n_treatment * X̄_control²)
        -- + X̄_treatment² * s²_control / (n_control * X̄_control⁴)
        SQRT(
            var_treatment / users_treatment / POWER(mean_control, 2)
            +
            POWER(mean_treatment, 2) * var_control
            / users_control
            / POWER(mean_control, 4)
        ) AS se_relative_effect,

        -- Standard error of the absolute difference of means:
        --
        -- SE(X̄_treatment - X̄_control)
        -- = sqrt(s²_control / n_control + s²_treatment / n_treatment)
        SQRT(
            var_control / users_control
            +
            var_treatment / users_treatment
        ) AS se_diff

    FROM stats
),

z_stats AS (
    SELECT
        *,

        -- Absolute two-sample z-statistic:
        --
        -- H0: X̄_treatment - X̄_control = 0
        --
        -- Z = (X̄_treatment - X̄_control) / SE
        (mean_treatment - mean_control) / NULLIF(se_diff, 0) AS z_value

    FROM calc
)

SELECT
    events_control,
    events_treatment,

    users_control,
    users_treatment,

    --mean_control,
    --mean_treatment,

    relative_effect,

    -- Approximate 95% CI for relative effect:
    -- uplift ± 1.96 * SE(uplift)
    relative_effect - 1.96 * se_relative_effect AS ci_lower,
    relative_effect + 1.96 * se_relative_effect AS ci_upper,

    z_value

FROM z_stats;

,events_control,events_treatment,users_control,users_treatment,relative_effect,ci_lower,ci_upper,z_value
0,13279.0,15082.0,12569,12431,0.148387,0.112103,0.18467,8.575708


### Interpretation of the z-value (two-sided test)

| Significance level ($\alpha$) | Critical value |
|:-----------------------------:|:--------------:|
| 0.05 | $|Z| > 1.96$ |
| 0.01 | $|Z| > 2.576$ |
| 0.003 | $|Z| > 2.968$ |

In general, the two-sided *p*-value is computed as

$$
p\text{-value}
=
2\left(1 - \Phi\left(|Z|\right)\right),
$$

where $\Phi(\cdot)$ is the cumulative distribution function (CDF) of the standard normal distribution.